# Shivaa Jewels — Final Reel Render (Google Colab)

**Kya karta hai:** aapki Drive wali video (`100601_1791300779339.mp4`) ko directly padhta hai,
asli boli transcribe karta hai, captions ko asli timings par set karta hai, premium
maroon-gold captions/graphics burn karta hai, aur final Reel **aapki Drive me** save karta hai.

**Privacy:** sab kuch aapke apne Google account ke Colab runtime me chalta hai; video kahin upload nahi hoti.

**Kaise chalayein:** upar menu me **Runtime → Run all** dabaiye. Drive mount ki permission maange to Allow kar dijiye.
Har cell ke niche result dikhega. Total ~4-6 minute.

**Output:** `MyDrive/ShivaaReel/shivaa-jewels-final.mp4` + `shivaa-jewels-captions.srt` + `.ass`


In [ ]:
# CELL 1 — Drive mount kijiye (Allow permission)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 — video dhoondho (144 MB preferred; warna 739 MB original)
import glob, os
CAND = ['100601_1791300779339.mp4', '100601_1791298723591.mp4']
hits = []
for c in CAND:
    hits += glob.glob(f'/content/drive/MyDrive/**/{c}', recursive=True)
if not hits:
    big = [p for p in glob.glob('/content/drive/MyDrive/**/*.mp4', recursive=True)
           if os.path.getsize(p) > 50_000_000]
    hits = sorted(big, key=os.path.getsize)
print('MILI FILES:', *hits, sep='\n  ')
SRC = hits[0] if hits else ''
# agar auto na mile to neeche khud path likh dijiye, jaise:
# SRC = '/content/drive/MyDrive/WhatsApp Video/100601_1791300779339.mp4'
assert SRC, 'video nahi mili — upar SRC me path set kijiye'
print('USE HO RAHI HAI:', SRC, os.path.getsize(SRC)//1_000_000, 'MB')


In [ ]:
# CELL 3 — tools install (ek baar, ~1-2 min)
!pip -q install imageio-ffmpeg pillow faster-whisper


In [ ]:
# CELL 4 — edit kit laao (GitHub se, public repo)
!rm -rf /content/kitrepo
!git clone -q --depth 1 --filter=blob:none --sparse -b arena/91f946d1-shivaa-ecom https://github.com/theunreal8055-coder/Shivaa.ecom.git /content/kitrepo
!git -C /content/kitrepo sparse-checkout set marketing/ads/reel-edit-kit
KIT = '/content/kitrepo/marketing/ads/reel-edit-kit'
!ls "$KIT"


In [ ]:
# CELL 5 — asli boli transcribe kijiye (~1-2 min). Neeche poori transcript chhapegi.
from faster_whisper import WhisperModel
import json
model = WhisperModel('small', device='auto', compute_type='int8')
segs_iter, info = model.transcribe(SRC, language='hi', vad_filter=True)
segs = [{'start': round(s.start, 2), 'end': round(s.end, 2), 'text': s.text.strip()} for s in segs_iter]
json.dump(segs, open('/content/segments.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
for s in segs:
    print(f"{s['start']:6.2f} -> {s['end']:6.2f}  {s['text']}")
print('\n>> Is transcript ko copy karke Arena chat me bhej dijiye — main approved spelling se mila kar verify kar dunga.')


In [ ]:
# CELL 6 — captions ko asli timings par set karo + graphics regenerate karo
import shutil, subprocess, sys
shutil.copy(KIT + '/timeline.json', KIT + '/timeline.provisional-backup.json')
subprocess.run([sys.executable, KIT + '/retiming/align_transcript.py',
                '--segments', '/content/segments.json', '--timeline', KIT + '/timeline.json'], check=True)
subprocess.run([sys.executable, KIT + '/make_captions.py'], check=True)


In [ ]:
# CELL 7 — FINAL RENDER (~2-4 min). Output aapki Drive me: MyDrive/ShivaaReel/
import os, shutil, subprocess, sys
OUTDIR = '/content/drive/MyDrive/ShivaaReel'
os.makedirs(OUTDIR, exist_ok=True)
subprocess.run([sys.executable, KIT + '/render.py', SRC, OUTDIR + '/shivaa-jewels-final.mp4'], check=True)
shutil.copy(KIT + '/captions/shivaa-jewels-captions.srt', OUTDIR + '/')
shutil.copy(KIT + '/captions/shivaa-jewels-captions.ass', OUTDIR + '/')
print('\nFINAL VIDEO:', OUTDIR + '/shivaa-jewels-final.mp4')
print('Drive app me ShivaaReel folder kholiye — reel ready hai.')


In [ ]:
# CELL 8 (OPTIONAL) — 10 second cinematic outro
# EXT = duakan ki exterior photo ka Drive path (khali chhoda to placeholder lagega)
# VO  = owner ki awaaz me recording (outro/OUTRO-VO-SCRIPT.txt), optional
EXT = ''   # jaise: '/content/drive/MyDrive/ShivaaReel/shop-exterior.jpg'
VO  = ''   # jaise: '/content/drive/MyDrive/ShivaaReel/outro-vo.m4a'
cmd = ['bash', KIT + '/outro/render_outro.sh',
       EXT or (KIT + '/outro/placeholder-exterior.png'),
       OUTDIR + '/shivaa-jewels-outro-10s.mp4'] + ([VO] if VO else [])
subprocess.run(cmd, check=True)


## Ho gaya!
- `MyDrive/ShivaaReel/shivaa-jewels-final.mp4` → Instagram Reel par upload kijiye (1080×1920, 30fps).
- `shivaa-jewels-captions.srt / .ass` → backup captions, kisi bhi editor me import ho sakti hain.
- Transcript (Cell 5 ka output) Arena chat me paste kar dijiye — approved Hindi spelling se cross-check ho jayega.
- Outro me asli duakan ki photo lagani ho to Cell 8 me `EXT` path bhar kar dobara Run kijiye.
